In [ ]:
import numpy as np
import matplotlib.pyplot as plt



 

In [ ]:
# Constants
epsilon_0 = 8.854187817e-12  # Vacuum permittivity (F/m)
mu_0 = 4 * np.pi * 1e-7      # Vacuum permeability (H/m)
c = 1 / np.sqrt(epsilon_0 * mu_0)  # Speed of light (m/s)
q = 1.0e-9                   # Charge (Coulombs)

# Observation grid
x = np.linspace(-2, 2, 40)
y = np.linspace(-2, 2, 40)
X, Y = np.meshgrid(x, y)

Z = 0  # Observation plane at z = 0
obs_points = np.stack((X, Y, np.full_like(X, Z)), axis=-1)

In [ ]:


# Charge trajectory: moving along x-axis at constant velocity

def charge_position(t):
    v = 0.5 * c  # velocity of charge
    return np.array([v * t, 0, 0]), np.array([v, 0, 0])

# Compute retarded time numerically
def retarded_time(obs_point, t_obs):
    def f(t_ret):
        r_q, _ = charge_position(t_ret)
        R = np.linalg.norm(obs_point - r_q)
        return t_obs - t_ret - R / c

    # Use Newton-Raphson method
    t_ret = t_obs - np.linalg.norm(obs_point) / c
    for _ in range(10):
        r_q, _ = charge_position(t_ret)
        R = np.linalg.norm(obs_point - r_q)
        f_val = t_obs - t_ret - R / c
        df_dt = 1 + (np.dot(obs_point - r_q, charge_position(t_ret + 1e-9)[1]) / (R * c))
        t_ret += f_val / df_dt
    return t_ret

# Compute E and B fields using Liénard–Wiechert potentials
def compute_fields(obs_point, t_obs):
    t_ret = retarded_time(obs_point, t_obs)
    r_q, v_q = charge_position(t_ret)
    R_vec = obs_point - r_q
    R = np.linalg.norm(R_vec)
    n = R_vec / R
    beta = v_q / c
    gamma = 1 / np.sqrt(1 - np.dot(beta, beta))
    denom = (1 - np.dot(n, beta))**3 * R**2
    E = (q / (4 * np.pi * epsilon_0)) * ((1 - np.dot(beta, beta)) * (n - beta)) / denom
    B = np.cross(n, E) / c
    return E, B



In [ ]:
# Time snapshot

t_obs = 0.0

# Compute fields at each observation point

E_field = np.zeros_like(obs_points)

B_field = np.zeros_like(obs_points)

for i in range(obs_points.shape[0]):

    for j in range(obs_points.shape[1]):

        E, B = compute_fields(obs_points[i, j], t_obs)

        E_field[i, j] = E

        B_field[i, j] = B

# Plot electric field vectors

plt.figure(figsize=(8, 6))

plt.quiver(X, Y, E_field[:, :, 0], E_field[:, :, 1], color='blue', scale=1e-7)

plt.title("Electric Field Vectors from Moving Charge")

plt.xlabel("x (m)")

plt.ylabel("y (m)")

plt.grid(True)

plt.axis('equal')

plt.show()

What are the lienerd-wiechert retarded potentials for electric and magnetic fields? How can these be combined to compute the induction of magnetic and electric fields so as to simulate the propagation of an electromagnetic wave? What additional principles do we need to represent this e.g. the induction law

 

 

 

 

The Liénard–Wiechert potentials describe the retarded (causally delayed) electric and magnetic potentials generated by a moving point charge. These are fundamental in classical electrodynamics for modeling how electromagnetic fields propagate from sources that are not stationary.

🔹 Liénard–Wiechert Potentials

For a point charge ( q ) moving along a trajectory ( \mathbf{r}_q(t) ), the scalar potential ( \phi ) and vector potential ( \mathbf{A} ) at a field point ( \mathbf{r} ) and time ( t ) are:

Scalar Potential:

$ \phi(\mathbf{r}, t) = \frac{q}{4\pi\epsilon0} \left[ \frac{1}{(1 - \mathbf{n} \cdot \mathbf{v}/c) R} \right]{\text{ret}} $

Vector Potential:

$ \mathbf{A}(\mathbf{r}, t) = \frac{q \mathbf{v}}{4\pi\epsilon0 c} \left[ \frac{1}{(1 - \mathbf{n} \cdot \mathbf{v}/c) R} \right]{\text{ret}} $

Where:

( \mathbf{v} ) is the velocity of the charge at the retarded time ( t_r ),
( \mathbf{n} = \frac{\mathbf{r} - \mathbf{r}q(tr)}{|\mathbf{r} - \mathbf{r}q(tr)|} ),
( R = |\mathbf{r} - \mathbf{r}q(tr)| ),
The subscript “ret” means all quantities are evaluated at the retarded time ( tr ), which satisfies: $ tr = t - \frac{|\mathbf{r} - \mathbf{r}q(tr)|}{c} $
🔹 Electric and Magnetic Fields from Potentials

Once you have ( \phi ) and ( \mathbf{A} ), you can compute the fields using:

Electric field: $ \mathbf{E} = -\nabla \phi - \frac{\partial \mathbf{A}}{\partial t} $
Magnetic field: $ \mathbf{B} = \nabla \times \mathbf{A} $
These fields naturally include induction effects due to the time-varying potentials.

🔹 Simulating Electromagnetic Wave Propagation

To simulate EM wave propagation using these potentials:

Model the source: Define the trajectory ( \mathbf{r}_q(t) ) and velocity ( \mathbf{v}(t) ) of the charge.
Compute retarded time: Solve for ( t_r ) numerically.
Evaluate potentials: Use the Liénard–Wiechert formulas.
Compute fields: Use the above differential relations.
Apply Maxwell’s equations to verify consistency and simulate wave behavior.
🔹 Additional Principles Needed

To fully represent wave propagation and induction, you need:

Faraday’s Law (Induction): $ \nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t} $
Ampère–Maxwell Law: $ \nabla \times \mathbf{B} = \mu0 \epsilon0 \frac{\partial \mathbf{E}}{\partial t} + \mu_0 \mathbf{J} $
Continuity Equation: $ \nabla \cdot \mathbf{J} + \frac{\partial \rho}{\partial t} = 0 $
These ensure that the fields obey causality and conservation laws.

Would you like a Python simulation of a moving charge emitting EM waves using these principles? Or perhaps a visual diagram showing how the retarded potentials relate to the fields?

 

 

 

Can you extend the code so that it includes contributions from Faradays law of induction and the Ampere-Maxwell law of induction?

 

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

# Constants

epsilon_0 = 8.854187817e-12  # Vacuum permittivity (F/m)

mu_0 = 4 * np.pi * 1e-7      # Vacuum permeability (H/m)

c = 1 / np.sqrt(epsilon_0 * mu_0)  # Speed of light (m/s)

q = 1.0e-9                   # Charge (Coulombs)

# Observation grid

x = np.linspace(-2, 2, 40)

y = np.linspace(-2, 2, 40)

X, Y = np.meshgrid(x, y)

Z = 0  # Observation plane at z = 0

obs_points = np.stack((X, Y, np.full_like(X, Z)), axis=-1)

# Charge trajectory: moving along x-axis at constant velocity

def charge_position(t):

    v = 0.5 * c  # velocity of charge

    return np.array([v * t, 0, 0]), np.array([v, 0, 0])

# Compute retarded time numerically

def retarded_time(obs_point, t_obs):

    def f(t_ret):

        r_q, _ = charge_position(t_ret)

        R = np.linalg.norm(obs_point - r_q)

        return t_obs - t_ret - R / c

    t_ret = t_obs - np.linalg.norm(obs_point) / c

    for _ in range(10):

        r_q, _ = charge_position(t_ret)

        R = np.linalg.norm(obs_point - r_q)

        f_val = t_obs - t_ret - R / c

        df_dt = 1 + (np.dot(obs_point - r_q, charge_position(t_ret + 1e-9)[1]) / (R * c))

        t_ret += f_val / df_dt

    return t_ret

# Compute E and B fields using Liénard–Wiechert potentials

def compute_fields(obs_point, t_obs):

    t_ret = retarded_time(obs_point, t_obs)

    r_q, v_q = charge_position(t_ret)

    R_vec = obs_point - r_q

    R = np.linalg.norm(R_vec)

    n = R_vec / R

    beta = v_q / c

    denom = (1 - np.dot(n, beta))**3 * R**2

    E = (q / (4 * np.pi * epsilon_0)) * ((1 - np.dot(beta, beta)) * (n - beta)) / denom

    B = np.cross(n, E) / c

    return E, B

# Time snapshots for derivatives

t_obs = 0.0

dt = 1e-10

# Compute fields at each observation point

E_field = np.zeros_like(obs_points)

B_field = np.zeros_like(obs_points)

E_field_dt = np.zeros_like(obs_points)

B_field_dt = np.zeros_like(obs_points)

for i in range(obs_points.shape[0]):

    for j in range(obs_points.shape[1]):

        E, B = compute_fields(obs_points[i, j], t_obs)

        E_dt, B_dt = compute_fields(obs_points[i, j], t_obs + dt)

        E_field[i, j] = E

        B_field[i, j] = B

        E_field_dt[i, j] = E_dt

        B_field_dt[i, j] = B_dt

# Time derivatives

dE_dt = (E_field_dt - E_field) / dt

dB_dt = (B_field_dt - B_field) / dt

# Compute curls using finite differences

def compute_curl(field, dx, dy):

    curl_z = np.zeros(field.shape[:2])

    for i in range(1, field.shape[0]-1):

        for j in range(1, field.shape[1]-1):

            dFy_dx = (field[i+1, j, 1] - field[i-1, j, 1]) / (2*dx)

            dFx_dy = (field[i, j+1, 0] - field[i, j-1, 0]) / (2*dy)

            curl_z[i, j] = dFy_dx - dFx_dy

    return curl_z

dx = x[1] - x[0]

dy = y[1] - y[0]

curl_E_z = compute_curl(E_field, dx, dy)

curl_B_z = compute_curl(B_field, dx, dy)

# Plotting

fig, axs = plt.subplots(2, 2, figsize=(12, 10))

axs[0, 0].quiver(X, Y, E_field[:, :, 0], E_field[:, :, 1], color='blue', scale=1e-7)

axs[0, 0].set_title("Electric Field Vectors")

axs[0, 0].set_xlabel("x (m)")

axs[0, 0].set_ylabel("y 